# CNN-regularised downward continuation (plug-and-play ADMM)

ADMM with a learned prior. Each iteration takes a data-consistency step
$x = \tilde z - \lambda (A\tilde z - y)$ with $\tilde z = z - u$, then a denoising step $z = D(x + u)$, then a
dual update $u \leftarrow u + x - z$. $D$ is a ConvNeXt U-Net denoiser trained on the 22,000 training patches with Gaussian
noise (σ ∈ [0, 50]/255). Settings: 100 iterations, λ = 2. Weights: `weights/pnp_denoiser.pth`.

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # make the gravdc package importable

import numpy as np
import torch

from gravdc import evaluate, format_summary, load_model, load_test_set, save_result, summarize
from gravdc.plotting import show_reconstruction
from gravdc.table3 import compare

DATA_DIR = "../data/test"
RESULTS_DIR = "../results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.9.0+cu128


## Data
Observation: the ground-level field continued 800 m upward in a single step (zero padding to 36 × 36).

In [2]:
from gravdc import upward_continue

names, x = load_test_set(DATA_DIR)
x = x.to(device)
y = upward_continue(x, 800)
print(len(names), "patches:", names)
from gravdc.sampling import pnp_admm

10 patches: ['data14.mat', 'data16.mat', 'data18.mat', 'data20.mat', 'data22.mat', 'data24.mat', 'data26.mat', 'data28.mat', 'data30.mat', 'data32.mat']


## Model and inference

In [3]:
denoiser = load_model("pnp_denoiser", device)
x_hat = pnp_admm(y, denoiser, height=800.0, iterations=100, lam=2.0)

## Metrics

In [4]:
per_image = evaluate(x, x_hat)
compare("CNN-regularized", summarize(per_image))
save_result("CNN-regularized", per_image, names, RESULTS_DIR)

CNN-regularized
  metric             this run              Table 3
  PSNR       28.3958 ± 1.7156     28.3958 ± 1.7156  ✓
  SSIM        0.9254 ± 0.0316      0.9254 ± 0.0316  ✓
  RE          0.1888 ± 0.0293      0.1888 ± 0.0293  ✓
  RMSE        0.0663 ± 0.0158      0.0663 ± 0.0158  ✓


In [5]:
# The field data cannot be redistributed, so figures of them are off by default (executed notebooks can then
# be shared without them). Set SHOW_FIGURE = True to look at a reconstruction.
SHOW_FIGURE = False
if SHOW_FIGURE:
    k = 0
    show_reconstruction(x[k], y[k], {"CNN-regularized": x_hat[k]}, title=names[k])